Importing the dataset thoroughly. 

In [ ]:
import pandas as pd

In [ ]:
books_initial = pd.read_csv('books.csv', engine='python', on_bad_lines='warn')
print(books_initial.columns.tolist())
print(books_initial.shape)

Importing as a dataframe with read_csv alone gives an error due to formatting issues with the dataset. When using python engine with warnings instead of errors that stop the import, we can import the dataset and see which lines were problematic. As it parses and skips, it affects the lines for indexing. Before coding further, the CSV was inspected in excel, where the actual problematic lines were discovered to ensure no indexing errors. 

We got two errors, 4 lines for an additional column, and 4 lines with quotation issues. In the raw CSV, the problematic lines with an added column are 3350, 4704, 5879, 8981, which are due to commas in the author name column, leading to the other columns being incorrect by 1 column and adding a 13th, titleless column. The quotation issues come from quotes in the book titles, these were lines 1571, 4514, 9967, 10870. 

In [ ]:
books = pd.read_csv('books.csv', on_bad_lines='warn') # create our dataframe with pandas
books.columns = books.columns.str.strip() # remove extra spaces in column names, notably num_pages

We were warned that 4 rows were skipped when we used pandas, only those with the extra column. This allows us to manually check the titles with quotes and we can edit those as needed. 

In [ ]:
with open('books.csv', 'r', encoding='utf-8') as f:
    quote_lines = f.readlines()

# check the problematic lines (subtract 1 for 0-indexing)
for i in [1570, 4513, 9966, 10869]:
    print(f"Line {i+1}: {quote_lines[i]}")

We want to make sure the titles include the proper quotations. 
The index was manually confirmed to ensure the correct row is being modified. The indexes are different from list above due to skipped rows from importing the dataset and the removed header row.

In [ ]:
books.loc[1569, 'title'] = """"Stand Back" Said the Elephant  "I'm Going to Sneeze!\""""
books.loc[4511, 'title'] = "\"Why Are All The Black Kids Sitting Together in the Cafeteria?\": A Psychologist Explains the Development of Racial Identity"
books.loc[9961, 'title'] = "\"Dear Genius...\": A Memoir of My Life with Truman Capote"
books.loc[10864, 'title'] = "\"A\" Is for Abductive : The Language of the Emerging Church"

Now let's inspect the rows with the extra commas, and the we will manually create a temporary dataframe with the corrected data and add it to our books df. 

In [ ]:
with open('books.csv', 'r', encoding='utf-8') as f:
    comma_lines = f.readlines()

# check the problematic lines (subtract 1 for 0-indexing)
for i in [3349, 4703, 5878, 8980]:
    print(f"Line {i+1}: {comma_lines[i]}")

In [ ]:
bad_lines = [3349, 4703, 5878, 8980]
manual_rows = []
with open('books.csv', 'r', encoding='utf-8') as f:
    lines = f.readlines()
    for i in bad_lines:
        line = lines[i].strip()
        parts = line.split(',', 12)
        if len(parts) == 13:
            parts = [parts[0], parts[1], parts[2] + parts[3], parts[4], parts[5], parts[6], parts[7], parts[8], parts[9], parts[10], parts[11], parts[12]]
        manual_rows.append(parts)

manual_df = pd.DataFrame(manual_rows, columns=books.columns)
manual_df # ensure it has the correct data

In [ ]:
# We confirmed the data is correct, so we can concatonate the manual df to books.
books = pd.concat([books, manual_df], ignore_index=True)
print(books.shape) # Print shape to confirm it is the expected 11127 rows, 12 columns.